# Sidorenko's Conjecture

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 26. Sidorenko's Conjecture

### Detailed Problem Description
A <em>graphon</em> is a symmetric measurable function $W \colon [0,1]^2 \to [0,1]$.  Given a graphon $W$ and a finite graph $H = (V(H),E(H))$, the homomorphism density $t(H,W)$ is defined as
$$ t(H,W) = \int_{[0,1]^{V(H)}} \prod_{\{v,w\} \in E(H)} W(x_v,x_w)\ \prod_{v \in V(H)} dx_v.$$
For a finite bipartite graph $H$, let $C(H)$ denote the least constant for which
$$t(H, W) \geq t(K_2, W)^{C(H)}$$
holds for all graphons $W$, where $K_2$ is the complete graph on two vertices.  What is $C(H)$?


## AlphaEvolve Search Configuration

**Prompt**

Sidorenko's conjecture

Act as a research mathematician and optimization specialist.

GOAL:
For the bipartite graph H (K_5,5 minus a 10-cycle), your task is to find a graphon W(x, y): [0, 1]^2 -> [0, 1] that minimizes the ratio of the homomorphism density t(H, W) to t(K_2, W)^|E(H)|, attempting to find a counterexample (ratio < 1).

Specifically, the Python function you have to provide has the following
signature:

def get_graphon() -> callable

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which outputs the density ratio t(H, W) / t(K2, W)^|E(H)|.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as small as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
def constant_graphon(x, y):
    return 0.5

### Evolved Code by AlphaEvolve

In [ ]:
import numpy as np

def evolved_graphon(x, y):
    # A highly non-constant graphon tested by AlphaEvolve
    return 0.5 * (1.0 + 0.9 * np.cos(2 * np.pi * x) * np.cos(2 * np.pi * y))

### Evaluator Function

In [ ]:
from scipy.integrate import nquad

edges = [
    (0, 5), (0, 6), (0, 7), (0, 8),
    (1, 6), (1, 7), (1, 8), (1, 9),
    (2, 7), (2, 8), (2, 9), (2, 5),
    (3, 8), (3, 9), (3, 5), (3, 6),
    (4, 9), (4, 5), (4, 6), (4, 7)
]

def t_K2(W) -> float:
    val, _ = nquad(lambda x, y: W(x, y), [[0, 1], [0, 1]])
    return val

def t_H(W, edges) -> float:
    # Monte Carlo integration
    num_samples = 10000
    pts = np.random.rand(num_samples, 10)
    total_val = 0.0
    for sample in pts:
        product = 1.0
        for u, v in edges:
            product *= W(sample[u], sample[v])
        total_val += product
    return total_val / num_samples

### Data Verification and Results

In [ ]:
tk2 = t_K2(evolved_graphon)
th = t_H(evolved_graphon, edges)
num_edges = len(edges)
print(f"t(K2, W)^|E|: {tk2**num_edges:.6f}")
print(f"t(H, W):      {th:.6f}")
print("Sidorenko inequality holds:", th >= (tk2 ** num_edges) - 1e-5)